MILESTONE : 1

# Calculate the frequency distribution of the correct  answer  (A, B, C, D, E) in train.csv. Based on your counts, what is the sum of the occurrences of the most frequent option and the least frequent option? 

In [4]:
import pandas as pd

train = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")

freq = train["answer"].value_counts()

print(freq)

most_freq = freq.max()
least_freq = freq.min()

print("Answer =", most_freq + least_freq)

answer
B    490
C    459
A    369
D    358
E    324
Name: count, dtype: int64
Answer = 814


# After converting the prompt column to lowercase and removing all standard punctuation characters (using Python's string.punctuation), split the text by whitespace. What is the total number of unique words (vocabulary size) across the entire cleaned prompt column of train.csv? 

In [5]:
import pandas as pd
import string

train = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")

trans = str.maketrans('', '', string.punctuation)

vocab = set()

for text in train["prompt"].fillna("").astype(str):
    cleaned = text.lower().translate(trans)
    vocab.update(cleaned.split())

print(len(vocab))

859


# Using the cleaned prompt from Row ID 1, filter out the standard English stop words using sklearn.feature_extraction.text.ENGLISH_STOP_WORDS. How many words are left in the prompt for Row ID 1 after filtering? 

In [6]:
import pandas as pd
import string
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

train = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")

prompt = train.loc[train["id"] == 1, "prompt"].iloc[0]

cleaned = prompt.lower().translate(
    str.maketrans('', '', string.punctuation)
)

words = cleaned.split()

filtered_words = [
    w for w in words
    if w not in ENGLISH_STOP_WORDS
]

print(len(filtered_words))

13


# Fit a default TfidfVectorizer(stop_words='english') on a list containing all the combined text of the prompts and options in train.csv. What is the exact total number of feature columns (vocabulary size) generated by the vectorizer?  

In [7]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

train = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")

combined_text = (
    train["prompt"].fillna("") + " " +
    train["A"].fillna("") + " " +
    train["B"].fillna("") + " " +
    train["C"].fillna("") + " " +
    train["D"].fillna("") + " " +
    train["E"].fillna("")
).tolist()

vectorizer = TfidfVectorizer(stop_words="english")

X = vectorizer.fit_transform(combined_text)

print(X.shape)
print("Vocabulary Size =", len(vectorizer.vocabulary_))

(2000, 2762)
Vocabulary Size = 2762


# Using the TF-IDF vectorizer fitted in Question 3, calculate the cosine similarity between the prompt and option A strictly for Row ID 1. What is the resulting similarity score? (Round to 4 decimal places).  

In [8]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

train = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")

# Fit vectorizer from Question 3
texts = (
    train["prompt"].fillna("") + " " +
    train["A"].fillna("") + " " +
    train["B"].fillna("") + " " +
    train["C"].fillna("") + " " +
    train["D"].fillna("") + " " +
    train["E"].fillna("")
).tolist()

tfidf = TfidfVectorizer(stop_words="english")
tfidf.fit(texts)

row = train[train["id"] == 1].iloc[0]

prompt_vec = tfidf.transform([row["prompt"]])
A_vec = tfidf.transform([row["A"]])

sim = cosine_similarity(prompt_vec, A_vec)[0, 0]

print(round(sim, 4))

0.272


# Expand the logic from Question 4: For every row in train.csv, calculate the cosine similarity between the prompt and each of its 5 options .  Then calculate the percentage of instances where the option with the highest cosine similarity matches the correct answer.  

In [11]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

train = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")

# Question 4 vectorizer
texts = (
    train["prompt"].fillna("") + " " +
    train["A"].fillna("") + " " +
    train["B"].fillna("") + " " +
    train["C"].fillna("") + " " +
    train["D"].fillna("") + " " +
    train["E"].fillna("")
).tolist()

tfidf = TfidfVectorizer(stop_words="english")
tfidf.fit(texts)

labels = ["A", "B", "C", "D", "E"]

correct = 0

for _, row in train.iterrows():

    prompt_vec = tfidf.transform([str(row["prompt"])])

    sims = []

    for opt in labels:
        opt_vec = tfidf.transform([str(row[opt])])
        sim = cosine_similarity(prompt_vec, opt_vec)[0, 0]
        sims.append(sim)

    predicted = labels[sims.index(max(sims))]

    if predicted == row["answer"]:
        correct += 1

percentage = 100 * correct / len(train)

print("Correct:", correct)
print("Percentage:", round(percentage, 4))

Correct: 271
Percentage: 13.55


# If the ground truth answer for a question is C, what is the MAP@3 score if a model predicts C A B?  

In [12]:
true_answer = "C"
prediction = ["C", "A", "B"]

score = 0

for i, pred in enumerate(prediction, start=1):
    if pred == true_answer:
        score = 1 / i
        break

print(score)

1.0


# If the ground truth answer for a question is  B, what is the MAP@3 score if a model predicts D B E?  

In [13]:
actual = "B"
predicted = ["D", "B", "E"]

for i, p in enumerate(predicted, start=1):
    if p == actual:
        map3 = 1 / i
        break
else:
    map3 = 0

print(map3)

0.5


# The Majority Class Baseline: Find the most frequent correct answer in the training set (using your data from Q1). Make a static prediction for every single row where that most frequent answer is your 1st guess, followed by the second most frequent, and then the third most frequent. What is the overall MAP@3 score of this "Majority Class" baseline on train.csv?

In [15]:
import pandas as pd
from collections import Counter

train = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")

# Frequency of answers
freq = train["answer"].value_counts()
print(freq)

top3 = freq.index[:3].tolist()
print("Top 3:", top3)

def apk(actual, predicted):
    for i, p in enumerate(predicted[:3]):
        if p == actual:
            return 1.0 / (i + 1)
    return 0.0

scores = [apk(ans, top3) for ans in train["answer"]]

map3 = sum(scores) / len(scores)

print("MAP@3 =", round(map3, 6))

answer
B    490
C    459
A    369
D    358
E    324
Name: count, dtype: int64
Top 3: ['B', 'C', 'A']
MAP@3 = 0.42125


# The TF-IDF Pipeline: Build a basic pipeline that evaluates every row in train.csv. For each row, calculate the TF-IDF cosine similarity between the prompt and each of the 5 options. Sort these options from highest similarity to lowest to form your top 3 predictions. What is the final average MAP@3 score of this TF-IDF pipeline across the entire training set?  


In [16]:
labels = ["A","B","C","D","E"]

scores = []

for _, row in train.iterrows():

    prompt_vec = tfidf.transform([row["prompt"]])

    sims = []

    for opt in labels:
        opt_vec = tfidf.transform([row[opt]])
        sims.append(cosine_similarity(prompt_vec, opt_vec)[0,0])

    ranked = [labels[i] for i in np.argsort(sims)[::-1]]

    # MAP@3
    actual = row["answer"]

    score = 0
    for j, pred in enumerate(ranked[:3], start=1):
        if pred == actual:
            score = 1 / j
            break

    scores.append(score)

map3 = np.mean(scores)
print(map3)

0.25525
